In [5]:
print(os.getcwd())
print(os.listdir('.'))
%cd ..

f:\Workspace\PYTHON\Projects\FeedbackAnalysis\notebooks
['acronyms_dictionary.txt', 'baseline_model.ipynb', 'kiem_tra_dau_tieng_viet.csv', 'main_PhoBERT_sentiment_Kaggle.ipynb', 'main_PhoBERT_topic_Kaggle.ipynb', 'PhoBERT_Data_Preprocessing.ipynb', 'PhoBERT_Inference.ipynb', 'vncorenlp']
f:\Workspace\PYTHON\Projects\FeedbackAnalysis


In [12]:
import os
import json

path = r"F:\Workspace\PYTHON\Projects\FeedbackAnalysis\phobert_sentiment_model"
config_path = os.path.join(path, "config.json")

print(f"Kiểm tra file: {config_path}")
print(f"File có tồn tại không? {os.path.exists(config_path)}")

if os.path.exists(config_path):
    with open(config_path, 'r') as f:
        data = json.load(f)
        print("Nội dung config.json:")
        print(json.dumps(data, indent=2))
        print(f"\nModel type tìm thấy: {data.get('model_type', 'KHÔNG CÓ model_type')}")

Kiểm tra file: F:\Workspace\PYTHON\Projects\FeedbackAnalysis\phobert_sentiment_model\config.json
File có tồn tại không? False


In [13]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, RobertaForSequenceClassification
import re
from underthesea import text_normalize

# =====================================================================
# 1. HỆ THỐNG TIỀN XỬ LÝ (Áp dụng cho câu nhập vào)
# =====================================================================
# Nạp từ điển (Đảm bảo file acronyms_dictionary.txt nằm đúng chỗ)
dict_path = "../acronyms_dictionary.txt"

def load_acronyms(file_path):
    acronyms_dict = {}
    if os.path.exists(file_path):
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'): continue
                if '=' in line:
                    key, value = line.split('=', 1)
                    acronyms_dict[key.strip().lower()] = value.strip().lower()
    return acronyms_dict

ACRONYMS_DICT = load_acronyms(dict_path)

# Hàm làm sạch chuyên biệt
def clean_sentence_for_test(text):
    if not isinstance(text, str) or str(text).strip() == "": return ""
    text = str(text).lower()
    text = text_normalize(text)
    
    if ACRONYMS_DICT:
        for shortcut, full_word in ACRONYMS_DICT.items():
            if re.match(r'^\w+$', shortcut, flags=re.UNICODE):
                text = re.sub(r'\b' + shortcut + r'\b', full_word, text)
            else:
                text = text.replace(shortcut, full_word)
                
    try:
        # Nhớ phải chạy Cell 1 để nạp rdrsegmenter vào RAM trước đó nhé
        sentences = rdrsegmenter.word_segment(text)
        text_segmented = sentences[0] if sentences else text
    except Exception:
        text_segmented = text
        
    return text_segmented

# =====================================================================
# 2. KHỞI TẠO BỘ NÃO KÉP (Nạp Model vào RAM)
# =====================================================================
print("⏳ Đang nạp hệ thống phân tích kép...")

# Tokenizer dùng chung
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")

# Đảm bảo đường dẫn này đúng với nơi bạn giải nén 2 thư mục
path_sentiment_model = "../models/sentiment_model" 
path_topic_model = "../models/sentiment_model"         

# Nạp Não 1
model_sentiment = RobertaForSequenceClassification.from_pretrained(path_sentiment_model,local_files_only=True)
model_sentiment.eval()

# Nạp Não 2
model_topic = RobertaForSequenceClassification.from_pretrained(path_topic_model,local_files_only=True)
model_topic.eval()

print("✅ Hệ thống đã sẵn sàng 100%!")

# =====================================================================
# 3. HÀM ĐIỀU PHỐI CHÍNH (The Facade)
# =====================================================================
def analyze_feedback(text):
    # Bước 1: Làm sạch câu nói
    clean_text = clean_sentence_for_test(text)
    
    # Bước 2: Mã hóa
    inputs = tokenizer(clean_text, return_tensors="pt", padding=True, truncation=True, max_length=256)
    
    # Bước 3: Dự đoán song song
    with torch.no_grad():
        output_sentiment = model_sentiment(**inputs)
        pred_sentiment_idx = torch.argmax(output_sentiment.logits, dim=1).item()
        label_sentiment = model_sentiment.config.id2label[pred_sentiment_idx]
        
        output_topic = model_topic(**inputs)
        pred_topic_idx = torch.argmax(output_topic.logits, dim=1).item()
        label_topic = model_topic.config.id2label[pred_topic_idx]
        
    return {
        "Raw": text,
        "Cleaned": clean_text,
        "Topic": label_topic,
        "Sentiment": label_sentiment
    }

⏳ Đang nạp hệ thống phân tích kép...


OSError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '../models/sentiment_model'. Use `repo_type` argument if needed.